# KannadaGPT-0.6B Inference

Run **KannadaGPT-0.6B** - A Kannada language model fine-tuned on Qwen3-0.6B using LoRA.

[![HuggingFace](https://img.shields.io/badge/HuggingFace-Model-yellow)](https://huggingface.co/Mithun501/KannadaGPT-0.6B)
[![GitHub](https://img.shields.io/badge/GitHub-Repo-blue)](https://github.com/mithun50/KannadaGPT-0.6B)

## 1. Install Dependencies

In [ ]:
!pip install -q transformers peft torch accelerate

import transformers
print(f"transformers: {transformers.__version__}")

## 2. Load Model

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Mithun501/KannadaGPT-0.6B"
BASE_MODEL = "Qwen/Qwen3-0.6B"

print(f"Loading base model: {BASE_MODEL}")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Loading LoRA adapter: {MODEL_ID}")
model = PeftModel.from_pretrained(base_model, MODEL_ID)

print(f"\nModel loaded on: {model.device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 3. Define Generation Function

In [ ]:
def generate_response(prompt, max_new_tokens=256, temperature=0.7, top_p=0.8):
    """
    Generate a response from KannadaGPT.
    
    Args:
        prompt: Input text in Kannada or English
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (higher = more creative)
        top_p: Nucleus sampling parameter
    
    Returns:
        Generated response text
    """
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract assistant's response
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    
    return response

print("generate_response() function ready!")

## 4. Test the Model

Try some Kannada prompts:

In [ ]:
# Test 1: Capital of India
prompt = "ಭಾರತದ ರಾಜಧಾನಿ ಯಾವುದು?"
print(f"Prompt: {prompt}")
print(f"\nResponse:\n{generate_response(prompt)}")

In [ ]:
# Test 2: Health tips
prompt = "ಆರೋಗ್ಯವಾಗಿರಲು ಮೂರು ಸಲಹೆಗಳನ್ನು ನೀಡಿ"
print(f"Prompt: {prompt}")
print(f"\nResponse:\n{generate_response(prompt)}")

In [ ]:
# Test 3: About Bangalore
prompt = "ಬೆಂಗಳೂರಿನ ಬಗ್ಗೆ ಹೇಳಿ"
print(f"Prompt: {prompt}")
print(f"\nResponse:\n{generate_response(prompt)}")

In [ ]:
# Test 4: Write a poem
prompt = "ಕನ್ನಡದಲ್ಲಿ ಕವಿತೆ ಬರೆಯಿರಿ"
print(f"Prompt: {prompt}")
print(f"\nResponse:\n{generate_response(prompt)}")

In [ ]:
# Test 5: Why does it rain?
prompt = "ಮಳೆ ಏಕೆ ಬರುತ್ತದೆ?"
print(f"Prompt: {prompt}")
print(f"\nResponse:\n{generate_response(prompt)}")

## 5. Interactive Chat

In [ ]:
# Interactive mode - enter your own prompts!
your_prompt = "ನಿಮ್ಮ ಪ್ರಶ್ನೆಯನ್ನು ಇಲ್ಲಿ ಬರೆಯಿರಿ"  # Replace with your question

print(f"Prompt: {your_prompt}")
print(f"\nResponse:\n{generate_response(your_prompt)}")

## Model Info

| Property | Value |
|----------|-------|
| **Base Model** | Qwen/Qwen3-0.6B |
| **Language** | Kannada (ಕನ್ನಡ) |
| **Fine-tuning** | LoRA (Low-Rank Adaptation) |
| **Training Data** | Cognitive-Lab/Kannada-Instruct-dataset |
| **Checkpoint** | 4500 / 48702 steps |

**Author**: [Mithun501](https://github.com/mithun50)